In [1]:
#import and setup
import pandas as pd
from pathlib import Path

df = pd.read_csv("../data/processed/accidents_clean.csv",
                 parse_dates=["date", "datetime"])
EXPORT = Path("../data/exports")
EXPORT.mkdir(parents=True, exist_ok=True)

print("Shape:", df.shape)

Shape: (20000, 33)


In [ ]:
# Export for Power BI
pbi_cols = [
    "accident_id","city","state","latitude","longitude",
    "datetime","date","hour","day_of_week","is_weekend",
    "road_type","lanes","traffic_signal","weather","visibility",
    "temperature","traffic_density","cause","accident_severity",
    "vehicles_involved","casualties","is_peak_hour",
    "festival","is_festival","risk_score",
    "month","month_name","quarter","year","is_fatal"
]

df[pbi_cols].to_csv(EXPORT / "accidents_pbi.csv", index=False)
print("Saved:", (EXPORT / "accidents_pbi.csv").resolve())

Saved: C:\Users\Himanshu\Documents\indian-road-accident-analysis\data\exports\accidents_pbi.csv


In [3]:
# Export KPI summary to Excel
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment

wb = Workbook()

# Style helpers
HEADER_FONT = Font(bold=True, color="FFFFFF")
HEADER_FILL = PatternFill("solid", fgColor="2E86AB")

def style_header(ws):
    for cell in ws[1]:
        cell.font = HEADER_FONT
        cell.fill = HEADER_FILL
        cell.alignment = Alignment(horizontal="center")

# ---------- Sheet 1: KPIs ----------
ws = wb.active
ws.title = "KPIs"
ws.append(["Metric", "Value"])
ws.append(["Total Accidents", len(df)])
ws.append(["Total Casualties", int(df["casualties"].sum())])
ws.append(["Avg Casualties per Accident", round(df["casualties"].mean(), 2)])
ws.append(["Avg Risk Score", round(df["risk_score"].mean(), 2)])
ws.append(["Fatal Accidents", int(df["is_fatal"].sum())])
ws.append(["Fatal %", round(100 * df["is_fatal"].mean(), 2)])
ws.append(["Weekend Accidents", int(df["is_weekend"].sum())])
ws.append(["Peak Hour Accidents", int(df["is_peak_hour"].sum())])
ws.append(["Festival Accidents", int(df["is_festival"].sum())])
style_header(ws)

# ---------- Sheet 2: By State ----------
ws2 = wb.create_sheet("By State")
ws2.append(["State", "Accidents", "Casualties", "Avg Risk"])
for state, g in df.groupby("state"):
    ws2.append([state, len(g), int(g["casualties"].sum()),
                round(g["risk_score"].mean(), 2)])
style_header(ws2)

# ---------- Sheet 3: By City ----------
ws3 = wb.create_sheet("By City")
ws3.append(["City", "State", "Accidents", "Casualties", "Avg Risk"])
for (city, state), g in df.groupby(["city", "state"]):
    ws3.append([city, state, len(g), int(g["casualties"].sum()),
                round(g["risk_score"].mean(), 2)])
style_header(ws3)

# ---------- Sheet 4: By Cause ----------
ws4 = wb.create_sheet("By Cause")
ws4.append(["Cause", "Accidents", "Casualties", "Avg Risk"])
for cause, g in df.groupby("cause"):
    ws4.append([cause, len(g), int(g["casualties"].sum()),
                round(g["risk_score"].mean(), 2)])
style_header(ws4)

# ---------- Sheet 5: By Weather ----------
ws5 = wb.create_sheet("By Weather")
ws5.append(["Weather", "Accidents", "Avg Risk"])
for w, g in df.groupby("weather"):
    ws5.append([w, len(g), round(g["risk_score"].mean(), 2)])
style_header(ws5)

# ---------- Sheet 6: By Visibility ----------
ws6 = wb.create_sheet("By Visibility")
ws6.append(["Visibility", "Accidents", "Avg Risk"])
for v, g in df.groupby("visibility"):
    ws6.append([v, len(g), round(g["risk_score"].mean(), 2)])
style_header(ws6)

# ---------- Sheet 7: By Hour ----------
ws7 = wb.create_sheet("By Hour")
ws7.append(["Hour", "Accidents", "Avg Risk"])
for hour, g in df.groupby("hour"):
    ws7.append([int(hour), len(g), round(g["risk_score"].mean(), 2)])
style_header(ws7)

# ---------- Sheet 8: Severity ----------
ws8 = wb.create_sheet("Severity")
ws8.append(["Severity", "Count", "%"])
for sev, cnt in df["accident_severity"].value_counts().items():
    ws8.append([sev, int(cnt), round(100 * cnt / len(df), 2)])
style_header(ws8)

wb.save(EXPORT / "kpi_summary.xlsx")
print("Saved:", (EXPORT / "kpi_summary.xlsx").resolve())

Saved: C:\Users\Himanshu\Documents\indian-road-accident-analysis\data\exports\kpi_summary.xlsx


In [8]:
#Verify exports
import os
files = sorted(os.listdir(EXPORT))
print(f"{len(files)} files in {EXPORT}:")
for f in files:
    size_kb = os.path.getsize(EXPORT / f) / 1024
    print(f" - {f}  ({size_kb:.1f} KB)")

2 files in ..\data\exports:
 - accidents_pbi.csv  (3552.1 KB)
 - kpi_summary.xlsx  (9.8 KB)


In [7]:
#Preview PBI feed
pbi = pd.read_csv(EXPORT / "accidents_pbi.csv")
print("PBI feed shape:", pbi.shape)
print("Columns:", list(pbi.columns))
pbi.head(3)

PBI feed shape: (20000, 30)
Columns: ['accident_id', 'city', 'state', 'latitude', 'longitude', 'datetime', 'date', 'hour', 'day_of_week', 'is_weekend', 'road_type', 'lanes', 'traffic_signal', 'weather', 'visibility', 'temperature', 'traffic_density', 'cause', 'accident_severity', 'vehicles_involved', 'casualties', 'is_peak_hour', 'festival', 'is_festival', 'risk_score', 'month', 'month_name', 'quarter', 'year', 'is_fatal']


,accident_id,city,state,latitude,longitude,datetime,date,hour,day_of_week,is_weekend,...,casualties,is_peak_hour,festival,is_festival,risk_score,month,month_name,quarter,year,is_fatal
0,0,Pune,Maharashtra,18.680827,73.930388,2023-10-22 05:00:00,2023-10-22,5,Sunday,True,...,2,False,NaN,False,0.85,10,Oct,4,2023,True
1,1,Mumbai,Maharashtra,18.817732,72.790846,2023-05-21 04:00:00,2023-05-21,4,Sunday,True,...,3,False,NaN,False,0.10,5,May,2,2023,False
2,2,Mumbai,Maharashtra,19.096889,72.819424,2024-07-10 13:00:00,2024-07-10,13,Wednesday,False,...,1,False,NaN,False,0.45,7,Jul,3,2024,False


In [ ]:
#Verify Excel sheets
from openpyxl import load_workbook
wb = load_workbook(EXPORT / "kpi_summary.xlsx")
print("Sheets:", wb.sheetnames)
print("\nKPI sheet preview:")
ws = wb["KPIs"]
for row in ws.iter_rows(values_only=True):
    print(row)

Sheets: ['KPIs', 'By State', 'By City', 'By Cause', 'By Weather', 'By Visibility', 'By Hour', 'Severity']

KPI sheet preview:
('Metric', 'Value')
('Total Accidents', 20000)
('Total Casualties', 34529)
('Avg Casualties per Accident', 1.73)
('Avg Risk Score', 0.44)
('Fatal Accidents', 2987)
('Fatal %', 14.94)
('Weekend Accidents', 5723)
('Peak Hour Accidents', 4948)
('Festival Accidents', 115)
